In [ ]:
import pandas as pd
import numpy as np
import json

from sklearn.preprocessing import RobustScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit

df = pd.read_csv('/content/methane-emissions.csv')


df.columns = [c.strip() for c in df.columns]

df = df.drop_duplicates()


df = df.sort_values(["Entity", "Year"])


possible_targets = [
    "Nitrogen oxides",
    "Nitrogen oxides emissions from waste",
    "nitrogen_oxides",
    "Methane emissions",
    "methane_emissions",
    "Methane",
]

target_col = None
for c in possible_targets:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    target_col = df.select_dtypes(include='number').columns[-1]


df = df.groupby("Entity").apply(lambda g: g.interpolate()).reset_index(drop=True)
df = df.fillna(method='bfill').fillna(method='ffill')

def remove_outliers(group, col):
    Q1 = group[col].quantile(0.25)
    Q3 = group[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    group[col] = np.clip(group[col], lower, upper)
    return group

df = df.groupby("Entity").apply(lambda g: remove_outliers(g, target_col)).reset_index(drop=True)
def fix_years(group):
    full_years = pd.DataFrame({
        "Year": np.arange(group["Year"].min(), group["Year"].max() + 1)
    })
    group = full_years.merge(group, on="Year", how="left")

    group["Entity"] = group["Entity"].fillna(method="ffill").fillna(method="bfill")
    group[target_col] = group[target_col].interpolate().fillna(method="bfill").fillna(method="ffill")

    return group

df = df.groupby("Entity").apply(fix_years).reset_index(drop=True)

WINDOW = 5
FUTURE_YEARS = 5
TEST_SIZE = 5
POLY_DEGREE = 2

N_SPLITS = 3

AUGMENT_TIMES = 3
NOISE_STD = 0.03

dropped_countries = []
predictions_list = []

for country in df['Entity'].unique():

    data = df[df['Entity'] == country].sort_values("Year").copy()

    if len(data) < 25:
        continue

    values = data[target_col].values.astype(float)
    years = data['Year'].values.astype(float).reshape(-1, 1)

    if len(values) <= (WINDOW + TEST_SIZE + N_SPLITS + 5):
        continue

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)

    fold_mae = []
    fold_rmse = []
    fold_acc = []

    rows_before_aug = None
    rows_after_aug = None

    for fold_i, (train_idx, test_idx) in enumerate(tscv.split(values), start=1):

        if len(test_idx) < TEST_SIZE:
            continue
        test_idx = test_idx[:TEST_SIZE]

        train_values = values[train_idx]
        train_years = years[train_idx]

        y_true = values[test_idx]
        test_years = years[test_idx]

        if len(train_values) <= WINDOW + 5:
            continue

        poly = PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)
        train_years_poly = poly.fit_transform(train_years)
        test_years_poly = poly.transform(test_years)

        trend_model = LinearRegression()
        trend_model.fit(train_years_poly, train_values)

        trend_train_pred = trend_model.predict(train_years_poly)
        trend_test_pred = trend_model.predict(test_years_poly)

        train_residuals = train_values - trend_train_pred

        scaler = RobustScaler()
        res_scaled = scaler.fit_transform(train_residuals.reshape(-1, 1)).flatten()

        X, y = [], []
        for i in range(WINDOW, len(res_scaled)):
            lag_features = res_scaled[i-WINDOW:i]
            X.append(lag_features)
            y.append(res_scaled[i])

        X = np.array(X)
        y = np.array(y)

        if len(X) < 10:
            continue

        if rows_before_aug is None:
            rows_before_aug = len(X)

        X_aug = [X]
        y_aug = [y]

        for _ in range(AUGMENT_TIMES):
            noise_x = np.random.normal(0, NOISE_STD, X.shape)
            noise_y = np.random.normal(0, NOISE_STD, y.shape)

            X_noisy = X + noise_x
            y_noisy = y + noise_y

            X_aug.append(X_noisy)
            y_aug.append(y_noisy)

        X_train_aug = np.vstack(X_aug)
        y_train_aug = np.hstack(y_aug)

        if rows_after_aug is None:
            rows_after_aug = len(X_train_aug)

        idx = np.random.permutation(len(X_train_aug))
        X_train_aug = X_train_aug[idx]
        y_train_aug = y_train_aug[idx]

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=42
        )

        model.fit(X_train_aug, y_train_aug)

        last_train_seq = res_scaled[-WINDOW:].copy()
        test_res_scaled_pred = []

        for _ in range(TEST_SIZE):
            pred = model.predict(last_train_seq.reshape(1, -1))[0]
            test_res_scaled_pred.append(pred)
            last_train_seq = np.append(last_train_seq[1:], pred)

        test_res_pred = scaler.inverse_transform(
            np.array(test_res_scaled_pred).reshape(-1, 1)
        ).flatten()

        final_pred = trend_test_pred + test_res_pred
        final_pred = np.clip(final_pred, 0, None)

        mae_fold = mean_absolute_error(y_true, final_pred)
        rmse_fold = np.sqrt(mean_squared_error(y_true, final_pred))

        wmape_fold = np.sum(np.abs(y_true - final_pred)) / (np.sum(np.abs(y_true)) + 1e-8)
        acc_fold = (1 - wmape_fold) * 100

        fold_mae.append(mae_fold)
        fold_rmse.append(rmse_fold)
        fold_acc.append(acc_fold)

    if len(fold_acc) == 0:
        continue

    mae = float(np.mean(fold_mae))
    rmse = float(np.mean(fold_rmse))
    accuracy = float(np.mean(fold_acc))

    if (
        round(accuracy, 6) == 100.0
        or accuracy < 80
        or np.isclose(mae, 0.0)
        or np.isclose(rmse, 0.0)
    ):
        dropped_countries.append({"country": country, "accuracy": float(accuracy)})
        continue

    poly_full = PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)
    full_years_poly = poly_full.fit_transform(years)

    trend_model_full = LinearRegression()
    trend_model_full.fit(full_years_poly, values)

    full_trend_pred = trend_model_full.predict(full_years_poly)
    full_residuals = values - full_trend_pred

    scaler_full = RobustScaler()
    res_scaled_full = scaler_full.fit_transform(full_residuals.reshape(-1, 1)).flatten()

    X_full, y_full = [], []
    for i in range(WINDOW, len(res_scaled_full)):
        X_full.append(res_scaled_full[i-WINDOW:i])
        y_full.append(res_scaled_full[i])

    X_full = np.array(X_full)
    y_full = np.array(y_full)

    if len(X_full) < 10:
        continue

    rows_before_aug_full = len(X_full)

    X_full_aug = [X_full]
    y_full_aug = [y_full]

    for _ in range(AUGMENT_TIMES):
        noise_x = np.random.normal(0, NOISE_STD, X_full.shape)
        noise_y = np.random.normal(0, NOISE_STD, y_full.shape)

        X_noisy = X_full + noise_x
        y_noisy = y_full + noise_y

        X_full_aug.append(X_noisy)
        y_full_aug.append(y_noisy)

    X_full_train = np.vstack(X_full_aug)
    y_full_train = np.hstack(y_full_aug)

    rows_after_aug_full = len(X_full_train)

    idx_full = np.random.permutation(len(X_full_train))
    X_full_train = X_full_train[idx_full]
    y_full_train = y_full_train[idx_full]

    model_full = RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    )

    model_full.fit(X_full_train, y_full_train)

    future_years = np.arange(
        data['Year'].max() + 1,
        data['Year'].max() + 1 + FUTURE_YEARS
    ).reshape(-1, 1)

    future_years_poly = poly_full.transform(future_years)
    trend_future = trend_model_full.predict(future_years_poly)

    last_seq = res_scaled_full[-WINDOW:].copy()
    future_res = []

    for _ in range(FUTURE_YEARS):
        pred = model_full.predict(last_seq.reshape(1, -1))[0]
        future_res.append(pred)
        last_seq = np.append(last_seq[1:], pred)

    future_res = scaler_full.inverse_transform(
        np.array(future_res).reshape(-1, 1)
    ).flatten()

    future_final = trend_future + future_res
    future_final = np.clip(future_final, 0, None)

    predictions_list.append({
        "country": country,
        "rows_before_augmentation": int(rows_before_aug if rows_before_aug is not None else 0),
        "rows_after_augmentation": int(rows_after_aug if rows_after_aug is not None else 0),
        "mae": float(mae),
        "rmse": float(rmse),
        "accuracy": float(accuracy),
        "future_predictions": [
            {
                "year": int(y),
                "predicted_methane_emissions": float(p)
            }
            for y, p in zip(future_years.flatten(), future_final)
        ]
    })

    print("=" * 50)
    print("Country:", country)
    print(f"Rows Before Augmentation: {rows_before_aug}")
    print(f"Rows After Augmentation: {rows_after_aug}")
    print(f"CV folds used: {len(fold_acc)}")
    print(f"MAE (CV avg)  : {mae:,.4f}")
    print(f"RMSE (CV avg) : {rmse:,.4f}")
    print(f"Accuracy (CV avg): {accuracy:.1f}%")

    print(pd.DataFrame({
        "Year": future_years.flatten(),
        "Predicted_metahne emissions": future_final
    }))

output_data = {
    "dropped_countries": dropped_countries,
    "predictions": predictions_list
}

with open('/content/methane_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print("❌ Dropped Countries:\n")

for c in dropped_countries:
    print(f"{c['country']} --> {c['accuracy']:.1f}%")

print(f"\nTotal Dropped: {len(dropped_countries)}")
print("\nJSON file saved to: /content/methane_predictions.json")

/tmp/ipykernel_4262/2984276916.py:46: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby("Entity").apply(lambda g: g.interpolate()).reset_index(drop=True)
/tmp/ipykernel_4262/2984276916.py:46: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby("Entity").apply(lambda g: g.interpolate()).reset_index(drop=True)
/tmp/ipykernel_4262/2984276916.py:46: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.groupby("Entity").apply(lambda g: g.interpolate()).reset_index(drop=True)
/tmp/ipykernel_4262/2984276916.py:46: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in

Country: Afghanistan
Rows Before Augmentation: 41
Rows After Augmentation: 164
CV folds used: 3
MAE (CV avg)  : 705,327.8341
RMSE (CV avg) : 858,242.4025
Accuracy (CV avg): 91.1%
   Year  Predicted_metahne emissions
0  2025                 1.594451e+07
1  2026                 1.606614e+07
2  2027                 1.618310e+07
3  2028                 1.631782e+07
4  2029                 1.644027e+07
Country: Africa
Rows Before Augmentation: 41
Rows After Augmentation: 164
CV folds used: 3
MAE (CV avg)  : 7,172,900.1120
RMSE (CV avg) : 7,801,172.1327
Accuracy (CV avg): 97.8%
   Year  Predicted_metahne emissions
0  2025                 1.426932e+09
1  2026                 1.447081e+09
2  2027                 1.468734e+09
3  2028                 1.490894e+09
4  2029                 1.513569e+09
Country: Albania
Rows Before Augmentation: 41
Rows After Augmentation: 164
CV folds used: 3
MAE (CV avg)  : 156,431.7255
RMSE (CV avg) : 178,361.4714
Accuracy (CV avg): 93.9%
   Year  Predicted_metah